# PSQ Exploratory Data Analysis

This notebook runs the project PSQ calculations with configurable paths. It summarizes missingness, distributions and descriptive correlations. No hypothesis tests or clinical categories are applied.

**Data access:** inside the project, the notebook uses `outputs/lemon/psychology/PSQ.csv`. For a standalone copy, place `PSQ.csv` beside the notebook or change `INPUT_FILE` below. Do not substitute a questionnaire with a different scoring scheme.

In [ ]:
from pathlib import Path
import sys, platform, importlib.metadata as metadata
print("Python:", platform.python_version())
for package in ["numpy", "scipy", "matplotlib"]:
    print(package, metadata.version(package))
cwd = Path.cwd()
project_root = next(
    (
        folder for folder in (cwd, *cwd.parents)
        if (folder / "psychology_eda/psq_eda.py").is_file()
    ),
    None,
)
if project_root is None:
    INPUT_FILE = cwd / "PSQ.csv"
    OUTPUT_DIR = cwd / "psq_results"
    SCRIPT = cwd / "psq_eda.py"
else:
    INPUT_FILE = project_root / "outputs/lemon/psychology/PSQ.csv"
    OUTPUT_DIR = INPUT_FILE.parent / "standalone"
    SCRIPT = project_root / "psychology_eda/psq_eda.py"

assert INPUT_FILE.is_file(), f"Set INPUT_FILE to your PSQ.csv path. Missing: {INPUT_FILE}"
assert SCRIPT.is_file(), f"Analysis script missing: {SCRIPT}"
print("Results folder:", OUTPUT_DIR)


## Run the analysis
The script validates participant IDs and score ranges, preserves missing entries, calculates quartiles and IQR, and creates figures. Correlations use complete cases across all five scores. The original 227-person file contains 220 complete records.

In [ ]:
import subprocess
result = subprocess.run([sys.executable, str(SCRIPT), "--input", str(INPUT_FILE),
                         "--output", str(OUTPUT_DIR)], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
result.check_returncode()


## Summary statistics and correlations

In [ ]:
import csv
for filename in ["PSQ_summary_statistics.csv", "PSQ_pearson_correlations.csv", "PSQ_spearman_correlations.csv"]:
    print("\n" + filename)
    with (OUTPUT_DIR / filename).open(newline="") as f:
        for row in csv.reader(f):
            print(" | ".join(row))


## Figures
Boxes show Q1–Q3. All observations are retained. Joy retains its supplied direction.

In [ ]:
from IPython.display import display, Image

figure_folders = [OUTPUT_DIR / "figures"]
if project_root is not None:
    demographics = (
        project_root / "data/lemon/source/Behavioural_Data_MPILMBB_LEMON"
        / "META_File_IDs_Age_Gender_Education_Drug_Smoke_SKID_LEMON.csv"
    )
    if demographics.is_file():
        project_figures = OUTPUT_DIR / "project_figures"
        subprocess.run(
            [
                sys.executable, "-m", "psychology_eda.psq_project_figures",
                "--input", str(INPUT_FILE), "--output", str(project_figures),
            ],
            cwd=project_root,
            check=True,
        )
        figure_folders.append(project_figures)
    else:
        print("The six original project figures require the LEMON demographics CSV.")

for folder in figure_folders:
    for figure in sorted(folder.glob("*.png")):
        print(figure.name)
        display(Image(filename=str(figure), width=900))


## Interpretation and limits
The mean total score describes the sample, not a percentage of stress. Correlations with the total score are partly due to its construction from the same subscales. This notebook does not test relationships with ECG or fMRI.

For the original project file, total mean ≈30.10, median ≈26.67 and IQR =20.00. The source CSV is not distributed inside this code package. The analysis.json output contains participant identifiers; keep it separate from public report files.